# 🍜 BeautifulSoup 기초

HTML/XML 문서에서 원하는 데이터를 쉽게 추출하는 파이썬 라이브러리입니다.

```bash
pip install beautifulsoup4
```

| 항목 | 내용 |
|------|------|
| **라이브러리** | beautifulsoup4 (bs4) |
| **용도** | HTML 파싱, 요소 검색, 데이터 추출 |
| **핵심 개념** | HTML → BeautifulSoup → 태그 검색 → 텍스트/속성 추출 |

---
## 1. HTML 파싱하기

`BeautifulSoup` 객체를 생성하면 HTML 문서를 파이썬에서 다룰 수 있는
트리 구조로 변환합니다.

In [ ]:
from bs4 import BeautifulSoup

# 예시 HTML 문서
html = '''
<html>
  <head><title>웹 스크래핑 예제</title></head>
  <body>
    <h1 class="main-title">BeautifulSoup 기초</h1>
    <div id="content" class="article">
      <p class="intro">첫 번째 문단입니다.</p>
      <p class="detail">두 번째 문단의 <strong>강조 텍스트</strong>입니다.</p>
      <a href="https://example.com" class="link-btn">링크 텍스트</a>
    </div>
    <ul id="menu">
      <li data-type="web">웹 스크래핑</li>
      <li data-type="api">API 호출</li>
      <li data-type="db">데이터베이스</li>
    </ul>
  </body>
</html>
'''

# HTML 파서로 BeautifulSoup 객체 생성
# 'html.parser'는 파이썬 내장 HTML 파서입니다
soup = BeautifulSoup(html, 'html.parser')

# 파싱 결과 확인
print(type(soup))  # BeautifulSoup 객체
print(f'제목 태그: {soup.title}')
print(f'제목 텍스트: {soup.title.string}')

---
## 2. 태그 검색 — find()와 find_all()

가장 많이 사용하는 두 가지 메서드입니다.

- `find()`: 조건에 맍는 **첫 번째** 요소 하나 반환
- `find_all()`: 조건에 맍는 **모든** 요소를 리스트로 반환

In [ ]:
# find() — 첫 번째 일치 요소
h1 = soup.find('h1')
print(f'태그명: {h1.name}')
print(f'텍스트: {h1.string}')
print(f'클래스: {h1["class"]}')
print(f'클래스(단일): {h1.get("class", [])}')

# find_all() — 모든 일치 요소
paragraphs = soup.find_all('p')
print(f'\n문단 개수: {len(paragraphs)}')
for i, p in enumerate(paragraphs):
    print(f'  [{i}] class={p.get("class")}, text="{p.text.strip()}"')

# 결과를 리스트로 변환하지 않고 슬라이싱도 가능
first_two = soup.find_all('p')[:2]
print(f'\n처음 2개: {[p.text.strip() for p in first_two]}')

---
## 3. CSS 선택자 — select()와 select_one()

CSS 선택자를 사용하면 더 유연하게 요소를 검색할 수 있습니다.

- `.class-name` → 클래스 선택
- `#id-name` → ID 선택
- `tag.class` → 태그 + 클래스
- `parent > child` → 직계 자식
- `parent descendant` → 모든 후손

In [ ]:
# select_one() — CSS 선택자로 첫 번째 요소
title = soup.select_one('h1.main-title')
print(f'CSS 선택자 결과: {title.text}')

# select() — CSS 선택자로 모든 요소
items = soup.select('ul#menu > li')
print(f'메뉴 항목: {[li.text for li in items]}')

# 속성 선택자
web_items = soup.select('li[data-type="web"]')
print(f'data-type=web: {[li.text for li in web_items]}')

# 중첩 선택
content = soup.select_one('div#content')
links = content.select('a')
print(f'콘텐츠 내 링크: {[a.text for a in links]}')

# 여러 클래스 동시 선택
intro = soup.select_one('p.intro')
detail = soup.select_one('p.detail')
print(f'intro: {intro.text.strip()}')
print(f'detail: {detail.text.strip()}')

---
## 4. 속성과 텍스트 추출

HTML 요소에서 속성(href, src, class 등)과 텍스트를 추출하는 방법입니다.

In [ ]:
# 링크 태그에서 속성 추출
link = soup.find('a')
print(f'href 속성: {link["href"]}')
print(f'class 속성: {link["class"]}')
print(f'전체 속성: {link.attrs}')

# 텍스트 추출 방법 비교
detail_p = soup.find('p', class_='detail')
print(f'\n.string: {detail_p.string}')          # 직계 텍스트만 (자식 태그 있으면 None)
print(f'.text: {detail_p.text}')                # 모든 자식 텍스트 합침
print(f'.get_text(strip=True): {detail_p.get_text(strip=True)}')  # 공백 제거

# 부모/형제/자식 탐색
strong = soup.find('strong')
print(f'\n부모 태그: {strong.parent.name}')
print(f'부모 텍스트: {strong.parent.text}')
print(f'이전 형제: {strong.previous_sibling.string}')

---
## 5. 실제 웹 페이지 파싱

`requests`로 웹 페이지를 가져오고 BeautifulSoup으로 파싱합니다.

In [ ]:
import requests
from bs4 import BeautifulSoup

# 실제 웹 페이지 가져오기
url = 'https://httpbin.org/html'
response = requests.get(url)
response.raise_for_status()

# BeautifulSoup으로 파싱
soup = BeautifulSoup(response.text, 'html.parser')

# h1 태그 찾기
h1 = soup.find('h1')
print(f'h1: {h1.text}')

# 모든 문단 찾기
paragraphs = soup.find_all('p')
for i, p in enumerate(paragraphs):
    print(f'p[{i}]: {p.text.strip()[:80]}')

---
## 6. 테이블 데이터 추출

HTML 테이블에서 행과 셀 데이터를 체계적으로 추출합니다.

In [ ]:
# HTML 테이블 파싱
table_html = '''
<table id="scores">
  <thead><tr><th>이름</th><th>과목</th><th>점수</th></tr></thead>
  <tbody>
    <tr><td>김철수</td><td>수학</td><td>92</td></tr>
    <tr><td>이영희</td><td>영어</td><td>88</td></tr>
    <tr><td>박지성</td><td>과학</td><td>95</td></tr>
  </tbody>
</table>
'''

soup = BeautifulSoup(table_html, 'html.parser')
table = soup.find('table')

# 헤더 추출
headers = [th.text for th in table.find_all('th')]
print(f'헤더: {headers}')

# 각 행의 데이터 추출
rows = []
for tr in table.find_all('tr')[1:]:  # 헤더 행 제외
    cells = [td.text for td in tr.find_all('td')]
    if cells:
        rows.append(cells)
        print(f'행: {cells}')

print(f'\n총 {len(rows)}행')

---
## 7. 데이터 정제 — 텍스트 다듬기

웹에서 가져온 텍스트는 불필요한 공백, 줄바꿈, HTML 엔티티가 포함되어 있을 수 있습니다.

In [ ]:
# 정제가 필요한 HTML 예시
messy_html = '''
<div class="article">
  <h2>  파이썬   웹 스크래핑  </h2>
  <p>이 문단은&nbsp;&nbsp;여러 공백과
     <br/>줄바꿈이 포함되어 있습니다.</p>
  <p>&lt;HTML 엔티티&gt;도 처리해야 합니다.</p>
</div>
'''

soup = BeautifulSoup(messy_html, 'html.parser')

# 정제 없이 추출
h2_raw = soup.find('h2').text
print(f'원본: "{h2_raw}"')

# get_text(strip=True)로 공백 정제
h2_clean = soup.find('h2').get_text(strip=True)
print(f'정제: "{h2_clean}"')

# HTML 엔티티 자동 변환
p2 = soup.find_all('p')[1]
print(f'엔티티 변환: "{p2.get_text(strip=True)}"')

# 여러 텍스트를 하나로 합치기
all_text = soup.find('div').get_text(separator=' ', strip=True)
print(f'전체: "{all_text}"')

---
## 🎯 연습 문제

배운 내용을 직접 실습해 보세요.

1. 다음 HTML에서 모든 링크(`<a>` 태그)의 `href` 속성과 텍스트를 추출하세요.
   ```html
   <nav><a href="/home">홈</a> | <a href="/about">소개</a> | <a href="/contact">연락처</a></nav>
   ```

2. CSS 선택자를 사용해 `class="highlight"`인 요소의 텍스트를 모두 추출하세요.
   ```html
   <p class="highlight">중요 내용</p><p>일반 내용</p><p class="highlight">강조 내용</p>
   ```

3. 다음 테이블에서 각 행의 데이터를 딕셔너리 리스트로 변환하세요.
   ```html
   <table><tr><th>이름</th><th>점수</th></tr><tr><td>김</td><td>90</td></tr><tr><td>이</td><td>85</td></tr></table>
   ```

4. `https://httpbin.org/html` 페이지에서 `<h1>` 태그의 텍스트를 추출하세요.

5. 중첩된 HTML에서 특정 `id`를 가진 요소 내의 모든 `<li>` 텍스트를 추출하세요.
   ```html
   <div id="menu"><ul><li>커피</li><li>차</li></ul></div>
   ```

In [ ]:
# 연습 문제 — 직접 코드를 작성해 보세요
from bs4 import BeautifulSoup

# 1. 링크 추출
html1 = '<nav><a href="/home">홈</a> | <a href="/about">소개</a> | <a href="/contact">연락처</a></nav>'

# 2. class 선택자
html2 = '<p class="highlight">중요 내용</p><p>일반 내용</p><p class="highlight">강조 내용</p>'

# 3. 테이블 → 딕셔너리 리스트
html3 = '<table><tr><th>이름</th><th>점수</th></tr><tr><td>김</td><td>90</td></tr><tr><td>이</td><td>85</td></tr></table>'

# 4. 실제 웹 페이지 h1 추출

# 5. 중첩 요소에서 li 추출
html5 = '<div id="menu"><ul><li>커피</li><li>차</li></ul></div>'

---
## 🎯 정리

| 기능 | 코드 | 설명 |
|------|------|------|
| 파싱 | `BeautifulSoup(html, 'html.parser')` | HTML을 파이썬 객체로 변환 |
| 태그 검색 | `soup.find('tag')` | 첫 번째 일치 요소 |
| 모두 검색 | `soup.find_all('tag')` | 모든 일치 요소 리스트 |
| CSS 선택자 | `soup.select('css')` | CSS 문법으로 검색 |
| CSS 하나 | `soup.select_one('css')` | CSS 문법으로 첫 번째 요소 |
| 속성 | `tag['href']` / `tag.attrs` | 태그 속성 값 접근 |
| 텍스트 | `tag.text` / `tag.string` | 태그 내 텍스트 |
| 정제 | `tag.get_text(strip=True)` | 공백 제거 텍스트 |
| 부모/자식 | `tag.parent` / `tag.children` | 트리 탐색 |